In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/the-vit-chennai-mess-food-predictor/sample_submission.csv
/kaggle/input/the-vit-chennai-mess-food-predictor/train.csv
/kaggle/input/the-vit-chennai-mess-food-predictor/test.csv


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

train = pd.read_csv("/kaggle/input/the-vit-chennai-mess-food-predictor/train.csv")
test = pd.read_csv("/kaggle/input/the-vit-chennai-mess-food-predictor/test.csv")
sample = pd.read_csv("/kaggle/input/the-vit-chennai-mess-food-predictor/sample_submission.csv")

X = train.drop("quality", axis=1)
y = train["quality"]

train.head()

train.info()
print("\nMissing values:\n", train.isnull().sum())
train.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    15000 non-null  int64  
 1   fixed acidity         15000 non-null  float64
 2   volatile acidity      15000 non-null  float64
 3   citric acid           15000 non-null  float64
 4   residual sugar        15000 non-null  float64
 5   chlorides             15000 non-null  float64
 6   free sulfur dioxide   15000 non-null  float64
 7   total sulfur dioxide  15000 non-null  float64
 8   density               15000 non-null  float64
 9   pH                    15000 non-null  float64
 10  sulphates             15000 non-null  float64
 11  alcohol               15000 non-null  float64
 12  quality               15000 non-null  float64
dtypes: float64(12), int64(1)
memory usage: 1.5 MB

Missing values:
 id                      0
fixed acidity           0
volat

,id,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.00000,15000.000000,15000.000000,15000.000000
mean,7499.500000,8.164143,0.505642,0.231830,2.187443,0.078725,13.204133,37.577600,0.996929,3.32146,0.617574,10.191986,5.652667
std,4330.271354,1.392511,0.134598,0.179609,0.519474,0.013835,7.922627,24.493564,0.001406,0.11653,0.108033,0.906488,0.800959
min,0.000000,4.100000,0.120000,0.000000,1.100000,0.012000,1.000000,4.000000,0.990640,2.74000,0.390000,8.200000,3.000000
25%,3749.750000,7.200000,0.400000,0.050000,1.900000,0.073000,6.000000,20.000000,0.995900,3.24000,0.550000,9.500000,5.000000
50%,7499.500000,7.800000,0.510000,0.240000,2.100000,0.078000,12.000000,31.000000,0.996800,3.32000,0.590000,9.800000,6.000000
75%,11249.250000,8.900000,0.600000,0.390000,2.300000,0.083000,17.000000,48.000000,0.997800,3.39000,0.660000,10.800000,6.000000
max,14999.000000,15.900000,1.580000,1.000000,13.400000,0.415000,53.000000,152.000000,1.003690,3.78000,2.000000,14.000000,8.000000


In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

X = train.drop(["quality", "id"], axis=1)
y = train["quality"]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=200, random_state=42)

model.fit(X_train, y_train)

y_pred = model.predict(X_valid)

rmse = mean_squared_error(y_valid, y_pred, squared=False)
print("Validation RMSE:", rmse)


Validation RMSE: 0.7455019841243438


In [4]:
X_test = test.drop("id", axis=1)

test_preds = model.predict(X_test)

submission = pd.DataFrame({
    "id": test["id"],
    "quality": test_preds
})


In [5]:
from sklearn.model_selection import GridSearchCV

rf = RandomForestRegressor(random_state=42)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10],
    'min_samples_split': [2, 5],
}

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV RMSE:", -grid_search.best_score_)


Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best Parameters: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 100}
Best CV RMSE: 0.7293032103036076


In [6]:
from sklearn.ensemble import RandomForestRegressor

best_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    random_state=42
)

best_model.fit(X, y)
X_test = test.drop("id", axis=1)
final_preds = best_model.predict(X_test)

submission_final = pd.DataFrame({
    "id": test["id"],
    "quality": final_preds
})
print(train.shape)
train.head()


(15000, 13)


,id,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,0,7.0,0.58,0.00,3.0,0.078,17.0,45.0,0.99492,3.36,0.93,10.5,5.0
1,1,7.8,0.55,0.12,7.5,0.087,6.0,17.0,0.99820,3.22,0.62,9.2,3.0
2,2,7.6,0.62,0.00,1.9,0.065,6.0,35.0,0.99640,3.58,0.77,9.4,6.0
3,3,10.2,0.32,0.49,2.6,0.066,5.0,20.0,0.99606,3.21,0.70,11.3,5.0
4,4,7.5,0.61,0.03,2.2,0.090,15.0,49.0,0.99560,3.53,0.58,9.7,5.0


In [7]:
submission_final.to_csv("submission_tuned.csv", index=False)
submission_final.head()

,id,quality
0,15000,5.452859
1,15001,5.743117
2,15002,5.369115
3,15003,5.265123
4,15004,5.702072
